# Bài tập: Hồi quy tuyến tính đa biến với Gradient Descent
## Bộ dữ liệu: California Housing

> **Thư viện sử dụng:** Chỉ dùng `numpy`, `pandas`, `matplotlib` — **không dùng sklearn hay bất kỳ thư viện AI/ML nào**.

**Nguồn dữ liệu:** Bộ dữ liệu California Housing (file CSV công khai). Dữ liệu được thu thập từ cuộc điều tra dân số Mỹ năm 1990 tại California.

**Thông tin chung:**
- **Số lượng mẫu:** ~20,640
- **Số đặc trưng đầu vào:** 8
- **Biến đầu ra:** median_house_value — giá trị liên tục (USD)

**Ý nghĩa các đặc trưng đầu vào:**
| STT | Tên đặc trưng | Ý nghĩa |
|-----|--------------|---------|
| 1 | longitude | Kinh độ |
| 2 | latitude | Vĩ độ |
| 3 | housing_median_age | Tuổi trung vị của nhà trong khu vực |
| 4 | total_rooms | Tổng số phòng trong khu vực |
| 5 | total_bedrooms | Tổng số phòng ngủ trong khu vực |
| 6 | population | Dân số của khu vực |
| 7 | households | Số hộ gia đình trong khu vực |
| 8 | median_income | Thu nhập trung vị của hộ gia đình |

**Biến đầu ra:** median_house_value — Giá trị trung vị của nhà trong khu vực (USD)

## a. Mô tả dữ liệu

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch


df = pd.read_csv("housing.csv")

# Bỏ cột phân loại 'ocean_proximity', chỉ giữ cột số
df = df.drop('ocean_proximity', axis=1)

# Tên các đặc trưng đầu vào (tất cả cột trừ biến đầu ra)
target_col = 'median_house_value'
feature_names = [c for c in df.columns if c != target_col]

print("=" * 60)
print("MÔ TẢ DỮ LIỆU CALIFORNIA HOUSING")
print("=" * 60)
print(f"Số lượng mẫu       : {df.shape[0]}")
print(f"Số đặc trưng đầu vào: {len(feature_names)}")
print(f"Biến đầu ra         : {target_col} (giá trị trung vị nhà, USD)")
print(f"Các đặc trưng       : {feature_names}")
print()
print("5 dòng đầu tiên:")
df.head()

In [ ]:
# Thống kê mô tả cơ bản
print("Thống kê mô tả:")
df.describe()

## b. Tiền xử lý dữ liệu
- Kiểm tra và xử lý dữ liệu thiếu (nếu có)
- Chia dữ liệu thành tập huấn luyện (training set) và tập kiểm tra (test set)

In [ ]:
# ---- Kiểm tra dữ liệu thiếu ----
print("Kiểm tra dữ liệu thiếu (Missing Values):")
print(df.isnull().sum())
print(f"\nTổng số giá trị thiếu: {df.isnull().sum().sum()}")
print("=> Không có dữ liệu thiếu, không cần xử lý.")

In [ ]:
# ---- Xử lý giá trị thiếu (nếu có) trước khi chia ----
df = df.dropna()
print(f"Số mẫu sau khi loại bỏ dòng thiếu: {df.shape[0]}")

# ---- Chia dữ liệu thành tập huấn luyện và tập kiểm tra (thủ công) ----
X = df.drop(target_col, axis=1).values  # Đặc trưng đầu vào
y = df[target_col].values                # Biến đầu ra

# Shuffle dữ liệu với seed cố định
np.random.seed(42)
indices = np.random.permutation(len(X))
test_size = int(len(X) * 0.2)

test_indices = indices[:test_size]
train_indices = indices[test_size:]

X_train, X_test = X[train_indices], X[test_indices]
y_train, y_test = y[train_indices], y[test_indices]

print(f"Kích thước tập huấn luyện: X_train = {X_train.shape}, y_train = {y_train.shape}")
print(f"Kích thước tập kiểm tra  : X_test  = {X_test.shape},  y_test  = {y_test.shape}")

## c. Chuẩn hóa đặc trưng đầu vào bằng Z-score

Công thức Z-score: 

$$z = \frac{x - \mu}{\sigma}$$

Trong đó:
- $x$: giá trị gốc
- $\mu$: giá trị trung bình (mean) — tính trên tập train
- $\sigma$: độ lệch chuẩn (std) — tính trên tập train

**Lưu ý:** Chỉ fit (tính $\mu$, $\sigma$) trên tập train, rồi transform cả train và test để tránh data leakage.

In [ ]:
# ---- Chuẩn hóa Z-score (thủ công) ----
# Tính mean và std trên tập train
mu = X_train.mean(axis=0)
sigma = X_train.std(axis=0)

# Chuẩn hóa
X_train_scaled = (X_train - mu) / sigma
X_test_scaled = (X_test - mu) / sigma

print("Sau khi chuẩn hóa Z-score:")
print(f"  X_train_scaled — Mean ≈ {X_train_scaled.mean(axis=0).round(6)}")
print(f"  X_train_scaled — Std  ≈ {X_train_scaled.std(axis=0).round(6)}")
print()

# Hiển thị dưới dạng DataFrame cho dễ đọc
df_scaled = pd.DataFrame(X_train_scaled[:5], columns=feature_names)
print("5 dòng đầu sau chuẩn hóa:")
print(df_scaled)

# ---- Chuyển sang PyTorch Tensor ----
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
X_test_tensor  = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor  = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

print(f"\nĐã chuyển sang PyTorch Tensor:")
print(f"  X_train_tensor: {X_train_tensor.shape}, dtype={X_train_tensor.dtype}")
print(f"  y_train_tensor: {y_train_tensor.shape}, dtype={y_train_tensor.dtype}")

## d. Xây dựng mô hình Hồi quy tuyến tính đa biến — Tối ưu bằng Gradient Descent (PyTorch)

**Mô hình hồi quy tuyến tính đa biến:**

$$\hat{y} = X \cdot \mathbf{w} + b$$

**Hàm mất mát (MSE):**

$$J(\mathbf{w}, b) = \frac{1}{2m} \sum_{i=1}^{m} (\hat{y}_i - y_i)^2$$

**Sử dụng PyTorch:**
- Khai báo `weights` và `bias` là `torch.tensor` với `requires_grad=True` để PyTorch tự tính gradient (autograd).
- Cập nhật tham số bằng Gradient Descent thủ công (không dùng `torch.optim`):

$$\mathbf{w} = \mathbf{w} - \alpha \cdot \frac{\partial J}{\partial \mathbf{w}}$$

$$b = b - \alpha \cdot \frac{\partial J}{\partial b}$$

In [ ]:
class LinearRegressionGD:
    """Hồi quy tuyến tính đa biến sử dụng Gradient Descent với PyTorch."""

    def __init__(self, n_features, learning_rate=0.01, n_iterations=1000):
        self.lr = learning_rate
        self.n_iter = n_iterations
        # Khởi tạo trọng số và bias bằng 0, bật autograd
        self.weights = torch.zeros(n_features, 1, requires_grad=True, dtype=torch.float32)
        self.bias = torch.zeros(1, requires_grad=True, dtype=torch.float32)
        self.loss_history = []

    def predict(self, X):
        """Dự đoán: y_hat = X @ w + b"""
        return X @ self.weights + self.bias

    def fit(self, X, y):
        """Huấn luyện mô hình bằng Gradient Descent (autograd)."""
        m = X.shape[0]
        self.loss_history = []

        for i in range(self.n_iter):
            # 1. Dự đoán
            y_hat = self.predict(X)

            # 2. Tính loss: J = (1/2m) * sum((y_hat - y)^2)
            loss = (1 / (2 * m)) * torch.sum((y_hat - y) ** 2)

            # 3. Tính gradient tự động bằng autograd
            loss.backward()

            # 4. Cập nhật tham số (tắt autograd khi cập nhật)
            with torch.no_grad():
                self.weights -= self.lr * self.weights.grad
                self.bias -= self.lr * self.bias.grad

            # 5. Reset gradient về 0
            self.weights.grad.zero_()
            self.bias.grad.zero_()

            # 6. Lưu loss
            self.loss_history.append(loss.item())

            # In loss mỗi 100 vòng
            if (i + 1) % 100 == 0:
                print(f"  Iteration {i+1:>5d}/{self.n_iter}  |  Loss = {loss.item():.4f}")

        print(f"\n=> Huấn luyện hoàn tất! Loss cuối cùng = {self.loss_history[-1]:.4f}")
        return self

In [ ]:
# ---- Huấn luyện mô hình ----
n_features = X_train_tensor.shape[1]
model = LinearRegressionGD(n_features, learning_rate=0.01, n_iterations=1000)
model.fit(X_train_tensor, y_train_tensor)

print(f"\nTrọng số (weights): {model.weights.detach().numpy().flatten().round(4)}")
print(f"Bias: {model.bias.item():.4f}")

In [ ]:
# ---- Vẽ đồ thị Loss qua các vòng lặp ----
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(model.loss_history) + 1), model.loss_history, color='blue', linewidth=1.5)
plt.xlabel('Số vòng lặp (Iterations)')
plt.ylabel('Loss (MSE / 2)')
plt.title('Đồ thị hàm mất mát theo số vòng lặp Gradient Descent')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## e. Đánh giá kết quả

Các chỉ số đánh giá:
- **MSE** (Mean Squared Error): $\frac{1}{m}\sum_{i=1}^{m}(\hat{y}_i - y_i)^2$
- **RMSE** (Root MSE): $\sqrt{MSE}$
- **MAE** (Mean Absolute Error): $\frac{1}{m}\sum_{i=1}^{m}|\hat{y}_i - y_i|$
- **R² Score**: $1 - \frac{\sum(\hat{y}_i - y_i)^2}{\sum(y_i - \bar{y})^2}$

In [ ]:
# ---- Đánh giá trên tập Train và Test ----

# Dự đoán (tắt autograd vì không cần tính gradient)
with torch.no_grad():
    y_train_pred = model.predict(X_train_tensor).numpy().flatten()
    y_test_pred  = model.predict(X_test_tensor).numpy().flatten()

# Hàm tính các chỉ số đánh giá thủ công (numpy)
def compute_mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def compute_mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def compute_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

def evaluate(y_true, y_pred, set_name):
    mse  = compute_mse(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = compute_mae(y_true, y_pred)
    r2   = compute_r2(y_true, y_pred)
    print(f"  [{set_name}]")
    print(f"    MSE  = {mse:.4f}")
    print(f"    RMSE = {rmse:.4f}")
    print(f"    MAE  = {mae:.4f}")
    print(f"    R²   = {r2:.4f}")
    print()
    return mse, rmse, mae, r2

print("=" * 50)
print("ĐÁNH GIÁ MÔ HÌNH HỒI QUY TUYẾN TÍNH (PyTorch GD)")
print("=" * 50)
train_metrics = evaluate(y_train, y_train_pred, "Tập huấn luyện (Train)")
test_metrics  = evaluate(y_test, y_test_pred, "Tập kiểm tra (Test)")

In [ ]:
# ---- Biểu đồ so sánh Giá trị thực vs Giá trị dự đoán (Tập Test) ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot: Thực tế vs Dự đoán
axes[0].scatter(y_test, y_test_pred, alpha=0.3, s=10, color='steelblue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
             'r--', linewidth=2, label='Đường lý tưởng (y = ŷ)')
axes[0].set_xlabel('Giá trị thực (y)')
axes[0].set_ylabel('Giá trị dự đoán (ŷ)')
axes[0].set_title('Giá trị thực vs Dự đoán (Tập Test)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Histogram phần dư (Residuals)
residuals = y_test - y_test_pred
axes[1].hist(residuals, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Phần dư (Residual = y - ŷ)')
axes[1].set_ylabel('Tần suất')
axes[1].set_title('Phân phối phần dư (Tập Test)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ---- Biểu đồ trọng số (Feature Importance) ----
plt.figure(figsize=(10, 5))
weights_np = model.weights.detach().numpy().flatten()
sorted_idx = np.argsort(np.abs(weights_np))[::-1]

colors = plt.cm.RdYlBu(np.linspace(0.1, 0.9, len(feature_names)))
plt.barh(range(len(feature_names)),
         weights_np[sorted_idx],
         color=colors)
plt.yticks(range(len(feature_names)),
           [feature_names[i] for i in sorted_idx])
plt.xlabel('Trọng số (Weight)')
plt.title('Trọng số của các đặc trưng trong mô hình hồi quy tuyến tính')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## Toàn bộ code gộp chung trong 1 cell

In [ ]:
# ============================================================
# TOÀN BỘ CODE: Hồi quy tuyến tính đa biến với Gradient Descent (PyTorch)
# Bộ dữ liệu: California Housing
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")

# ============================================================
# a. MÔ TẢ DỮ LIỆU
# ============================================================
df = pd.read_csv("housing.csv")
df = df.drop('ocean_proximity', axis=1)

target_col = 'median_house_value'
feature_names = [c for c in df.columns if c != target_col]

print("=" * 60)
print("MÔ TẢ DỮ LIỆU CALIFORNIA HOUSING")
print("=" * 60)
print(f"Số lượng mẫu       : {df.shape[0]}")
print(f"Số đặc trưng đầu vào: {len(feature_names)}")
print(f"Biến đầu ra         : {target_col} (giá trị trung vị nhà, USD)")
print(f"Các đặc trưng       : {feature_names}")
print()
print("5 dòng đầu tiên:")
print(df.head())
print()
print("Thống kê mô tả:")
print(df.describe())

# ============================================================
# b. TIỀN XỬ LÝ DỮ LIỆU
# ============================================================
# Kiểm tra dữ liệu thiếu
print("\n" + "=" * 60)
print("KIỂM TRA DỮ LIỆU THIẾU")
print("=" * 60)
print(df.isnull().sum())
print(f"\nTổng số giá trị thiếu: {df.isnull().sum().sum()}")

# Xử lý dữ liệu thiếu
df = df.dropna()
print(f"Số mẫu sau khi loại bỏ dòng thiếu: {df.shape[0]}")

# Chia dữ liệu thành tập huấn luyện và tập kiểm tra (thủ công)
X = df.drop(target_col, axis=1).values
y = df[target_col].values

np.random.seed(42)
indices = np.random.permutation(len(X))
test_size = int(len(X) * 0.2)

test_indices = indices[:test_size]
train_indices = indices[test_size:]

X_train, X_test = X[train_indices], X[test_indices]
y_train, y_test = y[train_indices], y[test_indices]

print(f"\nKích thước tập huấn luyện: X_train = {X_train.shape}, y_train = {y_train.shape}")
print(f"Kích thước tập kiểm tra  : X_test  = {X_test.shape},  y_test  = {y_test.shape}")

# ============================================================
# c. CHUẨN HÓA Z-SCORE
# ============================================================
print("\n" + "=" * 60)
print("CHUẨN HÓA Z-SCORE")
print("=" * 60)

mu = X_train.mean(axis=0)
sigma = X_train.std(axis=0)

X_train_scaled = (X_train - mu) / sigma
X_test_scaled = (X_test - mu) / sigma

print(f"X_train_scaled — Mean ≈ {X_train_scaled.mean(axis=0).round(6)}")
print(f"X_train_scaled — Std  ≈ {X_train_scaled.std(axis=0).round(6)}")
print()
print("5 dòng đầu sau chuẩn hóa:")
print(pd.DataFrame(X_train_scaled[:5], columns=feature_names))

# Chuyển sang PyTorch Tensor
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
X_test_tensor  = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor  = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

print(f"\nĐã chuyển sang PyTorch Tensor:")
print(f"  X_train_tensor: {X_train_tensor.shape}, dtype={X_train_tensor.dtype}")
print(f"  y_train_tensor: {y_train_tensor.shape}, dtype={y_train_tensor.dtype}")

# ============================================================
# d. XÂY DỰNG MÔ HÌNH HỒI QUY TUYẾN TÍNH — GRADIENT DESCENT (PyTorch)
# ============================================================

class LinearRegressionGD:
    """Hồi quy tuyến tính đa biến sử dụng Gradient Descent với PyTorch."""

    def __init__(self, n_features, learning_rate=0.01, n_iterations=1000):
        self.lr = learning_rate
        self.n_iter = n_iterations
        self.weights = torch.zeros(n_features, 1, requires_grad=True, dtype=torch.float32)
        self.bias = torch.zeros(1, requires_grad=True, dtype=torch.float32)
        self.loss_history = []

    def predict(self, X):
        return X @ self.weights + self.bias

    def fit(self, X, y):
        m = X.shape[0]
        self.loss_history = []

        for i in range(self.n_iter):
            y_hat = self.predict(X)
            loss = (1 / (2 * m)) * torch.sum((y_hat - y) ** 2)
            loss.backward()

            with torch.no_grad():
                self.weights -= self.lr * self.weights.grad
                self.bias -= self.lr * self.bias.grad

            self.weights.grad.zero_()
            self.bias.grad.zero_()
            self.loss_history.append(loss.item())

            if (i + 1) % 100 == 0:
                print(f"  Iteration {i+1:>5d}/{self.n_iter}  |  Loss = {loss.item():.4f}")

        print(f"\n=> Huấn luyện hoàn tất! Loss cuối cùng = {self.loss_history[-1]:.4f}")
        return self

# Huấn luyện mô hình
print("\n" + "=" * 60)
print("HUẤN LUYỆN MÔ HÌNH")
print("=" * 60)
n_features = X_train_tensor.shape[1]
model = LinearRegressionGD(n_features, learning_rate=0.01, n_iterations=1000)
model.fit(X_train_tensor, y_train_tensor)

print(f"\nTrọng số (weights): {model.weights.detach().numpy().flatten().round(4)}")
print(f"Bias: {model.bias.item():.4f}")

# ============================================================
# e. ĐÁNH GIÁ KẾT QUẢ
# ============================================================

# Dự đoán
with torch.no_grad():
    y_train_pred = model.predict(X_train_tensor).numpy().flatten()
    y_test_pred  = model.predict(X_test_tensor).numpy().flatten()

# Hàm tính các chỉ số đánh giá
def compute_mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def compute_mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def compute_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

def evaluate(y_true, y_pred, set_name):
    mse  = compute_mse(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = compute_mae(y_true, y_pred)
    r2   = compute_r2(y_true, y_pred)
    print(f"  [{set_name}]")
    print(f"    MSE  = {mse:.4f}")
    print(f"    RMSE = {rmse:.4f}")
    print(f"    MAE  = {mae:.4f}")
    print(f"    R²   = {r2:.4f}")
    print()
    return mse, rmse, mae, r2

print("\n" + "=" * 60)
print("ĐÁNH GIÁ MÔ HÌNH HỒI QUY TUYẾN TÍNH (PyTorch GD)")
print("=" * 60)
train_metrics = evaluate(y_train, y_train_pred, "Tập huấn luyện (Train)")
test_metrics  = evaluate(y_test, y_test_pred, "Tập kiểm tra (Test)")

# ============================================================
# VẼ ĐỒ THỊ
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Đồ thị Loss
axes[0, 0].plot(range(1, len(model.loss_history) + 1), model.loss_history, color='blue', linewidth=1.5)
axes[0, 0].set_xlabel('Số vòng lặp (Iterations)')
axes[0, 0].set_ylabel('Loss (MSE / 2)')
axes[0, 0].set_title('Đồ thị hàm mất mát theo số vòng lặp GD')
axes[0, 0].grid(True, alpha=0.3)

# 2. Scatter: Thực tế vs Dự đoán
axes[0, 1].scatter(y_test, y_test_pred, alpha=0.3, s=10, color='steelblue')
axes[0, 1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
                'r--', linewidth=2, label='Đường lý tưởng (y = ŷ)')
axes[0, 1].set_xlabel('Giá trị thực (y)')
axes[0, 1].set_ylabel('Giá trị dự đoán (ŷ)')
axes[0, 1].set_title('Giá trị thực vs Dự đoán (Tập Test)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Histogram phần dư
residuals = y_test - y_test_pred
axes[1, 0].hist(residuals, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[1, 0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[1, 0].set_xlabel('Phần dư (Residual = y - ŷ)')
axes[1, 0].set_ylabel('Tần suất')
axes[1, 0].set_title('Phân phối phần dư (Tập Test)')
axes[1, 0].grid(True, alpha=0.3)

# 4. Biểu đồ trọng số
weights_np = model.weights.detach().numpy().flatten()
sorted_idx = np.argsort(np.abs(weights_np))[::-1]
colors = plt.cm.RdYlBu(np.linspace(0.1, 0.9, len(feature_names)))
axes[1, 1].barh(range(len(feature_names)), weights_np[sorted_idx], color=colors)
axes[1, 1].set_yticks(range(len(feature_names)))
axes[1, 1].set_yticklabels([feature_names[i] for i in sorted_idx])
axes[1, 1].set_xlabel('Trọng số (Weight)')
axes[1, 1].set_title('Trọng số của các đặc trưng')
axes[1, 1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()